# Silver temperature

Validate Frost temperature observations while preserving their sensor height, sampling frequency, parallel sensor and quality metadata.

This notebook contains the same functions as `silver_layer/temperature.py`.

This run cover all stations in the Bronze.

## 1. Schemas and implementation

Raw scalar values remain available for inspection. The full business key retains the source's sensor and series distinctions.


In [1]:
"""Validate Frost observations without mixing sensors or sampling frequencies.

Frost time and quality definitions:
https://frost.met.no/concepts2.html
https://frost.met.no/dataclarifications.html
"""

import hashlib
import json
import os
import re
import shutil
from datetime import date, datetime, timezone
from decimal import Decimal
from pathlib import Path
from uuid import uuid4

from pyspark.sql import Window, functions as F

RULE_VERSION = "temperature-v1"
RESOLUTIONS = {"PT1M": 60, "PT10M": 600, "PT30M": 1800,
               "PT1H": 3600, "PT3H": 10800, "PT6H": 21600}
SERIES_KEY = ["source_id", "element_id", "time_resolution", "time_offset",
              "time_series_id", "level_type", "level_unit", "level_value"]
BUSINESS_KEY = SERIES_KEY + ["observed_utc"]
RAW_SCHEMA = """
    sourceId STRING, referenceTime STRING,
    observations ARRAY<STRUCT<elementId: STRING, value: STRING, unit: STRING,
        timeOffset: STRING, timeResolution: STRING, timeSeriesId: STRING,
        qualityCode: STRING, performanceCategory: STRING, exposureCategory: STRING,
        level: STRUCT<levelType: STRING, unit: STRING, value: STRING>>>,
    source_file STRING, source_sha256 STRING, retrieval_manifest STRING,
    retrieved_at_utc STRING, requested_station STRING,
    period_start STRING, period_end STRING, record_index LONG
"""
SERIES_SCHEMA = """
    source_id STRING, element_id STRING, time_resolution STRING, time_offset STRING,
    time_series_id LONG, level_type STRING, level_unit STRING, level_value DECIMAL(12,4),
    first_second LONG, last_second LONG, step_seconds LONG
"""



from tempfile import TemporaryDirectory
from pyspark.sql import SparkSession


### Verify and stage inputs


In [2]:
def save_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{uuid4().hex}.tmp")
    try:
        temporary.write_text(json.dumps(payload, indent=2) + "\n")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)



In [3]:
def epoch(value):
    """Require an explicit offset before interpreting a source timestamp."""
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.utcoffset() is None:
        raise ValueError("Timestamp must contain an offset")
    return int(parsed.timestamp())



In [4]:
def monthly_periods(start, end):
    """Use complete UTC months to match the existing Bronze request folders."""
    first, limit = date.fromisoformat(start), date.fromisoformat(end)
    if first >= limit or first.day != 1 or limit.day != 1:
        raise ValueError("Use month boundaries with start before exclusive end")
    periods = []
    while first < limit:
        following = date(first.year + (first.month == 12), first.month % 12 + 1, 1)
        periods.append((str(first), str(following)))
        first = following
    return periods



In [5]:
def validate_stations(stations):
    if not stations or any(not re.fullmatch(r"SN[0-9]+", s) for s in stations):
        raise ValueError("Supply one or more station IDs such as SN18700")
    return sorted(set(stations))



In [6]:
def load_response(folder, station, period=None):
    """Verify the latest retrieval. Corrupt latest inputs never fall back silently."""
    candidates = []
    for manifest in sorted(folder.glob("retrieval_*.metadata.json")):
        metadata = json.loads(manifest.read_text())
        epoch(metadata["retrieved_at_utc"])
        retrieved = datetime.fromisoformat(metadata["retrieved_at_utc"].replace("Z", "+00:00"))
        candidates.append((retrieved, str(manifest), metadata))
    if not candidates:
        raise ValueError("Missing retrieval manifest")
    _, manifest, metadata = max(candidates, key=lambda item: (item[0], item[1]))
    if metadata["source"] != "Frost" or metadata["station_id"] != station:
        raise ValueError("Unexpected source or station in manifest")
    if period and (metadata["period_start"], metadata["period_end_exclusive"]) != period:
        raise ValueError("Manifest does not match the requested period")
    path = (folder / metadata["response_file"]).resolve()
    if path.parent != folder.resolve():
        raise ValueError("Response path escapes its Bronze folder")
    body = path.read_bytes()
    if hashlib.sha256(body).hexdigest() != metadata["sha256"]:
        raise ValueError("Response checksum mismatch")
    if len(body) != metadata["size_bytes"]:
        raise ValueError("Response size mismatch")
    # Decode decimal tokens as strings to avoid losing precision during staging.
    payload = json.loads(body, parse_float=str)
    if not isinstance(payload, dict) or payload.get("apiVersion") != "v0":
        raise ValueError("Expected a Frost v0 response object")
    rows = payload.get("data")
    status, http_status = metadata["status"], metadata["http_status"]
    if status == "empty" and http_status in (404, 412):
        error = payload.get("error", {})
        if str(error.get("code")) != str(http_status):
            raise ValueError("No-data error code does not match HTTP status")
        if http_status == 412 and not error.get("reason", "").startswith("No time series found for this combination of parameters"):
            raise ValueError("Unexpected HTTP 412 error")
        if rows:
            raise ValueError("No-data response unexpectedly contains measurements")
        rows = []
    elif http_status == 200 and status in ("success", "empty"):
        if not isinstance(rows, list) or payload.get("error"):
            raise ValueError("Expected a data array without an API error")
        if (payload.get("currentItemCount") != len(rows)
                or payload.get("totalItemCount") != len(rows)
                or payload.get("offset") != 0 or payload.get("nextLink")):
            raise ValueError("Incomplete or inconsistent response pagination")
        if (status == "empty") != (len(rows) == 0):
            raise ValueError("Manifest status does not match response data")
    else:
        raise ValueError("Unexpected retrieval status")
    lineage = {
        "source_file": str(path), "source_sha256": metadata["sha256"],
        "retrieval_manifest": manifest, "retrieved_at_utc": metadata["retrieved_at_utc"],
        "requested_station": station, "status": status,
    }
    return rows, lineage



In [7]:
def stage_month(bronze_root, staging, start, end, stations):
    """Validate one response at a time and stage one parent record per line."""
    staging = Path(staging)
    staging.mkdir(parents=True, exist_ok=False)
    selected, failures = [], []
    paths = []
    for station in validate_stations(stations):
        folder = Path(bronze_root) / "observations" / f"station={station}" / f"period={start}_to_{end}"
        target = staging / f"{station}.jsonl"
        try:
            rows, lineage = load_response(folder, station, (start, end))
            observations = 0
            # Reject malformed parent structures as an input failure, not an empty month.
            # Missing scalar values are kept for Spark's row-level quarantine.
            with target.open("w") as handle:
                for index, record in enumerate(rows):
                    if not isinstance(record, dict) or not isinstance(record.get("observations"), list) or not record["observations"]:
                        raise ValueError("Missing, malformed or empty observations array")
                    for observation in record["observations"]:
                        if not isinstance(observation, dict):
                            raise ValueError("Observation must be an object")
                        level = observation.get("level")
                        if level is not None and not isinstance(level, dict):
                            raise ValueError("Sensor level must be an object")
                        for key, value in observation.items():
                            if key != "level" and isinstance(value, (dict, list)):
                                raise ValueError(f"Unexpected nested observation field: {key}")
                        if level and any(isinstance(v, (dict, list)) for v in level.values()):
                            raise ValueError("Sensor level fields must be scalar")
                    if any(isinstance(record.get(k), (dict, list)) for k in ("sourceId", "referenceTime")):
                        raise ValueError("Source identifier and timestamp must be scalar")
                    observations += len(record["observations"])
                    staged = dict(record)
                    staged.update(lineage)
                    staged.update(period_start=start, period_end=end, record_index=index)
                    handle.write(json.dumps(staged, separators=(",", ":")) + "\n")
            selected.append({**lineage, "source_records": len(rows), "source_observations": observations})
            if rows:
                paths.append(str(target.resolve()))
        except (OSError, ValueError, KeyError, TypeError) as exc:
            target.unlink(missing_ok=True)
            failures.append({"station_id": station, "period_start": start, "error": str(exc)})
    return paths, selected, failures



### Read, normalize and classify observations


In [8]:
def read_temperature(spark, paths):
    """Expand observations while retaining parent identity and input lineage."""
    if paths:
        raw = spark.read.schema(RAW_SCHEMA).option("mode", "FAILFAST").json(paths)
    else:
        raw = spark.createDataFrame([], RAW_SCHEMA)
    return raw.select("*", F.posexplode("observations").alias("observation_index", "observation")).drop("observations")



In [9]:
def normalize_temperature(raw):
    """Keep raw values alongside UTC timestamps, exact decimals and quality flags."""
    rows = raw.select(
        "source_file", "source_sha256", "retrieval_manifest", "retrieved_at_utc",
        "requested_station", "period_start", "period_end", "record_index", "observation_index",
        F.col("sourceId").alias("source_id"), F.col("referenceTime").alias("source_time"),
        F.col("observation").alias("source_observation"),
        F.col("observation.elementId").alias("element_id"),
        F.col("observation.timeResolution").alias("time_resolution"),
        F.col("observation.timeOffset").alias("time_offset"),
        F.col("observation.unit").alias("unit"),
        F.col("observation.level.levelType").alias("level_type"),
        F.col("observation.level.unit").alias("level_unit"),
        F.col("observation.performanceCategory").alias("performance_category"),
        F.col("observation.exposureCategory").alias("exposure_category"),
    )
    rows = (
        rows.withColumn("station_id", F.regexp_extract("source_id", r"^(SN[0-9]+):[0-9]+$", 1))
        .withColumn("observed_utc", F.try_to_timestamp("source_time"))
        .withColumn("temperature_c", F.expr("try_cast(source_observation.value AS DECIMAL(16,6))"))
        .withColumn("level_value", F.expr("try_cast(source_observation.level.value AS DECIMAL(12,4))"))
        .withColumn("time_series_id", F.expr("try_cast(source_observation.timeSeriesId AS BIGINT)"))
        .withColumn("quality_code", F.expr("try_cast(source_observation.qualityCode AS INT)"))
        .withColumn("quality_uncertain", F.col("quality_code").isin(2, 4))
        .withColumn("local_date", F.to_date(F.from_utc_timestamp("observed_utc", "Europe/Oslo")))
    )
    # Subdaily referenceTime already is the observation time. Never add timeOffset.
    checks = [
        (F.col("source_time").rlike(r"(Z|[+-][0-9]{2}:[0-9]{2})$") & F.col("observed_utc").isNotNull(), "invalid_timestamp"),
        (F.col("source_id").rlike(r"^SN[0-9]+:[0-9]+$") & (F.col("station_id") == F.col("requested_station")), "invalid_source_id"),
        ((F.col("observed_utc") >= F.to_timestamp("period_start")) & (F.col("observed_utc") < F.to_timestamp("period_end")), "outside_requested_month"),
        (F.col("element_id") == "air_temperature", "unexpected_element"),
        (F.col("unit") == "degC", "missing_or_unexpected_unit"),
        (F.col("temperature_c").isNotNull(), "invalid_temperature"),
        (F.col("source_observation.value").rlike(r"^[+-]?[0-9]+(?:\.[0-9]{1,6}0*)?$"), "temperature_precision_or_format"),
        (F.col("time_resolution").isin(*RESOLUTIONS), "unsupported_resolution"),
        (F.col("time_offset").rlike(r"^PT(?:[0-9]+H)?(?:[0-9]+M)?(?:[0-9]+S)?$") & (F.col("time_offset") != "PT"), "invalid_offset"),
        ((F.col("time_series_id") >= 0) & F.col("source_observation.timeSeriesId").rlike(r"^[0-9]+$"), "invalid_series_id"),
        ((F.col("level_type") == "height_above_ground") & (F.col("level_unit") == "m") & (F.col("level_value") >= 0)
         & F.col("source_observation.level.value").rlike(r"^[+-]?[0-9]+(?:\.[0-9]{1,4}0*)?$"), "invalid_sensor_level"),
        (F.col("source_observation.qualityCode").isin("0", "1", "2", "4"), "unaccepted_quality_code"),
    ]
    # Keep every applicable reason so a missing unit cannot hide another issue.
    reasons = []
    for condition, reason in checks:
        reasons.append(F.when(~F.coalesce(condition, F.lit(False)), F.lit(reason)))
    return rows.withColumn("issues", F.filter(F.array(*reasons), lambda issue: issue.isNotNull()))



In [10]:
def classify_temperature(rows):
    """Keep distinct series and account for duplicates without choosing conflicting values."""
    invalid = rows.filter(F.size("issues") > 0).withColumn("disposition_reason", F.lit("invalid_observation"))
    valid = rows.filter(F.size("issues") == 0)
    # Include quality and sensor metadata when deciding if two observations agree.
    value = F.struct("temperature_c", "unit", "quality_code", "performance_category", "exposure_category")
    variants = valid.groupBy(*BUSINESS_KEY).agg(F.countDistinct(value).alias("_variants"))
    inspected = valid.join(variants, BUSINESS_KEY)
    conflicts = inspected.filter("_variants > 1").drop("_variants")
    conflicts = conflicts.withColumn("disposition_reason", F.lit("conflicting_observations"))
    copies = Window.partitionBy(*BUSINESS_KEY).orderBy("source_file", "record_index", "observation_index")
    equivalent = inspected.filter("_variants = 1").drop("_variants").withColumn("_copy", F.row_number().over(copies))
    accepted = equivalent.filter("_copy = 1").drop("_copy").withColumn("disposition_reason", F.lit(None).cast("string"))
    duplicates = equivalent.filter("_copy > 1").drop("_copy").withColumn("disposition_reason", F.lit("duplicate_observation"))
    return accepted, invalid.unionByName(conflicts), duplicates



### Assess nominal coverage using series metadata


In [11]:
def duration_seconds(value):
    match = re.fullmatch(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", value)
    if not match or not any(match.groups()):
        raise ValueError("Unsupported duration")
    hours, minutes, seconds = match.groups()
    return int(hours or 0) * 3600 + int(minutes or 0) * 60 + int(seconds or 0)



In [12]:
def load_series(bronze_root, stations, start, end):
    """Read validity metadata for nominal coverage, without assuming every station is hourly."""
    first, limit = epoch(start + "T00:00:00Z"), epoch(end + "T00:00:00Z")
    intervals, selected, failures = [], [], []
    for station in validate_stations(stations):
        folder = Path(bronze_root) / "series" / f"station={station}"
        try:
            records, lineage = load_response(folder, station)
            station_intervals = []
            for record in records:
                if record.get("elementId") != "air_temperature":
                    continue
                lower = max(first, epoch(record["validFrom"]))
                upper = min(limit, epoch(record["validTo"]) if record.get("validTo") else limit)
                if lower >= upper:
                    continue
                step = RESOLUTIONS[record["timeResolution"]]
                offset = duration_seconds(record["timeOffset"])
                source = record["sourceId"]
                if not re.fullmatch(station + r":[0-9]+", source):
                    raise ValueError("Series source does not match requested station")
                level = record["level"]
                height = Decimal(str(level["value"]))
                if not height.is_finite() or height != height.quantize(Decimal("0.0001")) or abs(height) >= 100000000:
                    raise ValueError("Invalid series height")
                first_tick = lower + (offset - lower) % step
                if first_tick < upper:
                    station_intervals.append((source, "air_temperature", record["timeResolution"],
                        record["timeOffset"], int(record["timeSeriesId"]), level["levelType"], level["unit"],
                        height, first_tick, upper - 1, step))
            intervals.extend(station_intervals)
            selected.append(lineage)
        except (OSError, ValueError, KeyError, TypeError, ArithmeticError) as exc:
            failures.append({"station_id": station, "error": str(exc)})
    return intervals, selected, failures



In [13]:
def coverage_report(spark, accepted, intervals):
    """Report nominal sampling gaps, not invented observations or hourly completeness.

    Validity windows are treated as [validFrom, validTo). Frequencies can vary
    within a series, so nominal gaps are diagnostics rather than publication gates.
    """
    schedules = spark.createDataFrame(intervals, SERIES_SCHEMA)
    expected = schedules.select(
        *SERIES_KEY,
        F.explode(F.sequence("first_second", "last_second", "step_seconds")).alias("second"),
    ).select(*SERIES_KEY, F.col("second").cast("timestamp").alias("observed_utc")).distinct()
    observed = accepted.select(*BUSINESS_KEY)
    missing = expected.join(observed, BUSINESS_KEY, "left_anti")
    unexpected = observed.join(expected, BUSINESS_KEY, "left_anti")
    report = expected.groupBy(*SERIES_KEY).count().withColumnRenamed("count", "nominal_expected")
    actual = observed.groupBy(*SERIES_KEY).count().withColumnRenamed("count", "accepted_observations")
    report = report.join(actual, SERIES_KEY, "full")
    report = report.join(missing.groupBy(*SERIES_KEY).count().withColumnRenamed("count", "nominal_missing"), SERIES_KEY, "left")
    report = report.join(unexpected.groupBy(*SERIES_KEY).count().withColumnRenamed("count", "outside_nominal_schedule"), SERIES_KEY, "left")
    return report.fillna(0, subset=["accepted_observations", "nominal_missing", "outside_nominal_schedule"])



### Process monthly batches and record outputs


In [14]:
def process_month(spark, bronze_root, batch, start, end, stations):
    """Write one bounded monthly batch and release staging only after Spark finishes."""
    batch = Path(batch)
    batch.mkdir(parents=True)
    staging = batch / "_staged"
    cached = []
    try:
        paths, selected, failures = stage_month(bronze_root, staging, start, end, stations)
        save_json(batch / "inputs.json", {"selected": selected, "failures": failures})
        normalized = normalize_temperature(read_temperature(spark, paths)).cache()
        cached.append(normalized)
        input_rows = normalized.count()
        expected_input = sum(item["source_observations"] for item in selected)
        if input_rows != expected_input:
            raise ValueError("Expanded Spark rows do not reconcile with source observations")
        accepted, quarantine, duplicates = classify_temperature(normalized)
        outputs = {"accepted": accepted, "quarantine": quarantine, "duplicates": duplicates}
        counts = {}
        for name, frame in outputs.items():
            frame.cache()
            cached.append(frame)
            counts[name] = frame.count()
        if sum(counts.values()) != input_rows:
            raise ValueError("Row dispositions do not reconcile")
        if accepted.groupBy(*BUSINESS_KEY).count().filter("count > 1").count():
            raise ValueError("Accepted observation keys are not unique")

        intervals, series_selected, series_failures = load_series(bronze_root, stations, start, end)
        save_json(batch / "series_inputs.json", {"selected": series_selected, "failures": series_failures})
        coverage = coverage_report(spark, accepted, intervals).cache()
        cached.append(coverage)
        issues = {}
        for row in quarantine.select(F.explode("issues").alias("issue")).groupBy("issue").count().collect():
            issues[row["issue"]] = row["count"]
        conflict_rows = quarantine.filter("disposition_reason = 'conflicting_observations'").count()
        coverage_counts = coverage.agg(F.sum("nominal_missing"), F.sum("outside_nominal_schedule")).first()
        quality = {
            "source_records": sum(item["source_records"] for item in selected),
            "input_observations": input_rows, "accepted_rows": counts["accepted"],
            "quarantined_rows": counts["quarantine"], "duplicate_rows": counts["duplicates"],
            "conflicting_rows": conflict_rows, "issue_counts": issues,
            "uncertain_accepted_rows": accepted.filter("quality_uncertain").count(),
            "negative_accepted_rows": accepted.filter("temperature_c < 0").count(),
            "requested_station_months": len(set(stations)),
            "empty_station_months": sum(item["status"] == "empty" for item in selected),
            "input_failures": len(failures), "series_metadata_failures": len(series_failures),
            "nominal_missing_observations": coverage_counts[0] or 0,
            "outside_nominal_schedule": coverage_counts[1] or 0,
        }
        for name, frame in outputs.items():
            frame.write.parquet(str(batch / name))
        coverage.write.parquet(str(batch / "coverage"))
        save_json(batch / "quality.json", quality)
        return quality
    finally:
        for frame in cached:
            frame.unpersist(blocking=True)
        if staging.exists():
            shutil.rmtree(staging)



In [15]:
def run(bronze_root, output_root, start, end, stations, spark):
    """Process full UTC months with a caller-owned Spark session and explicit station list."""
    periods = monthly_periods(start, end)
    stations = validate_stations(stations)
    bronze_root, output_root = Path(bronze_root).resolve(), Path(output_root).resolve()
    if output_root == bronze_root or bronze_root in output_root.parents:
        raise ValueError("Silver output must not be inside Bronze")
    output = output_root / uuid4().hex
    output.mkdir(parents=True)
    info = {"status": "started", "rule_version": RULE_VERSION,
            "started_at_utc": datetime.now(timezone.utc).isoformat(),
            "start_date": start, "end_date_exclusive": end, "date_timezone": "UTC",
            "stations": stations, "spark_version": spark.version,
            "scope_note": "Dates use UTC months, not Norwegian calendar-year boundaries.",
            "completed_months": []}
    save_json(output / "run.json", info)
    reports = []
    try:
        spark.conf.set("spark.sql.session.timeZone", "UTC")
        for first, limit in periods:
            print(f"Silver temperature: processing {first} to {limit}", flush=True)
            batch = output / f"month={first[:7]}"
            quality = process_month(spark, bronze_root, batch, first, limit, stations)
            reports.append({"month": first[:7], **quality})
            info["completed_months"].append(first[:7])
            save_json(output / "run.json", info)
        totals = {}
        for key in reports[0]:
            if isinstance(reports[0][key], int):
                totals[key] = sum(report[key] for report in reports)
        save_json(output / "quality.json", {"totals": totals, "months": reports})
        info["status"] = "partial" if any(totals[k] for k in ("quarantined_rows", "input_failures", "series_metadata_failures")) else "success"
        # Operational success never implies complete or geographically representative coverage.
        info["coverage_requires_review"] = any(totals[k] for k in ("empty_station_months", "nominal_missing_observations", "outside_nominal_schedule"))
    except BaseException as exc:
        info.update(status="failed", error=str(exc), error_type=type(exc).__name__)
        raise
    finally:
        info["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        save_json(output / "run.json", info)
    print(f"Silver temperature output: {output}")
    return output



## 2. Edit these settings

In [16]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise ValueError("Open this notebook from the repository root or notebooks folder")

bronze_root = PROJECT_ROOT / "data/bronze/frost"
output_root = PROJECT_ROOT / "data/silver/temperature"
start_date = "2024-01-01"
end_date = "2026-01-01"
stations = json.loads((bronze_root / "eligible-stations.json").read_text())

preview_station = "SN18700"
preview_start = "2024-07-01"
preview_end = "2024-08-01"

# Create a Spark session with sufficient memory and parallelism
spark = (
    SparkSession.builder
    .appName("Silver Temperature")
    .master("local[8]")
    .config("spark.driver.memory", "50g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/14 16:06:10 WARN Utils: Your hostname, homelab, resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/14 16:06:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/roland/UiS/H2026/group14/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/14 16:06:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Verify one source response

The manifest selects a specific retrieval and its checksum verifies the original bytes. No-data responses and input failures are recorded separately. JSON Lines staging changes the reading format without modifying Bronze. One nested parent record is written per line, so Spark still expands and transforms the observations.


In [17]:
preview_directory = TemporaryDirectory(prefix="silver-temperature-preview-")
paths, inputs, input_failures = stage_month(
    bronze_root, Path(preview_directory.name) / "staged",
    preview_start, preview_end, [preview_station],
)
print(json.dumps(inputs, indent=2))
print("Input failures:", input_failures)
raw = read_temperature(spark, paths)
raw.select("sourceId", "referenceTime", "observation").show(5, truncate=False)


[
  {
    "source_file": "/home/roland/UiS/H2026/group14/data/bronze/frost/observations/station=SN18700/period=2024-07-01_to_2024-08-01/response_b8e659374af2452b1eb595e5d0b0abbeb00d41427a5d0c33a98fc10561ef3283.json",
    "source_sha256": "b8e659374af2452b1eb595e5d0b0abbeb00d41427a5d0c33a98fc10561ef3283",
    "retrieval_manifest": "/home/roland/UiS/H2026/group14/data/bronze/frost/observations/station=SN18700/period=2024-07-01_to_2024-08-01/retrieval_20261030T131340040690Z_7e072b8d143d47c493dfcdeaa5633755.metadata.json",
    "retrieved_at_utc": "2026-10-30T13:13:40.040658+00:00",
    "requested_station": "SN18700",
    "status": "success",
    "source_records": 44640,
    "source_observations": 99696
  }
]
Input failures: []
+---------+------------------------+----------------------------------------------------------------------------------+
|sourceId |referenceTime           |observation                                                                       |
+---------+----------------

## 4. Normalize and inspect quality

Negative temperatures are valid. Missing units are not silently replaced with °C. Quality codes 0/1 are accepted, 2/4 remain accepted with an uncertainty flag, and other codes are quarantined under this project's policy. All applicable row issues are retained.


In [18]:
normalized = normalize_temperature(raw).cache()
normalized.select(
    "source_id", "observed_utc", "temperature_c", "level_value",
    "time_resolution", "quality_code", "quality_uncertain", "issues"
).show(10, truncate=False)
normalized.groupBy("quality_code", "unit").count().orderBy("quality_code").show()


26/11/14 16:06:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------------+-------------+-----------+---------------+------------+-----------------+------+
|source_id|observed_utc       |temperature_c|level_value|time_resolution|quality_code|quality_uncertain|issues|
+---------+-------------------+-------------+-----------+---------------+------------+-----------------+------+
|SN18700:0|2024-07-01 00:00:00|13.000000    |10.0000    |PT10M          |2           |true             |[]    |
|SN18700:0|2024-07-01 00:00:00|11.800000    |2.0000     |PT1M           |0           |false            |[]    |
|SN18700:0|2024-07-01 00:00:00|13.000000    |10.0000    |PT1H           |2           |true             |[]    |
|SN18700:0|2024-07-01 00:00:00|11.800000    |2.0000     |PT1H           |0           |false            |[]    |
|SN18700:0|2024-07-01 00:00:00|13.000000    |10.0000    |PT10M          |2           |true             |[]    |
|SN18700:0|2024-07-01 00:00:00|11.800000    |2.0000     |PT10M          |0           |false            |

## 5. Separate accepted records, quarantine and duplicate copies

Different heights and frequencies remain distinct observations. Equivalent copies collapse to one accepted row, while every removed copy remains in the duplicate output. Conflicting valid values for the same full key are quarantined.


In [19]:
accepted, quarantine, duplicates = classify_temperature(normalized)
accepted = accepted.cache()
quarantine = quarantine.cache()
duplicates = duplicates.cache()
input_count = normalized.count()
accepted_count = accepted.count()
quarantine_count = quarantine.count()
duplicate_count = duplicates.count()
assert input_count == accepted_count + quarantine_count + duplicate_count
print({"input": input_count, "accepted": accepted_count,
       "quarantine": quarantine_count, "duplicates": duplicate_count})
quarantine.select("source_time", "source_observation", "issues", "disposition_reason").show(10, truncate=False)
accepted.groupBy("level_value", "time_resolution").count().orderBy("level_value", "time_resolution").show()


{'input': 99696, 'accepted': 95232, 'quarantine': 0, 'duplicates': 4464}
+-----------+------------------+------+------------------+
|source_time|source_observation|issues|disposition_reason|
+-----------+------------------+------+------------------+
+-----------+------------------+------+------------------+

+-----------+---------------+-----+
|level_value|time_resolution|count|
+-----------+---------------+-----+
|     2.0000|          PT10M| 4464|
|     2.0000|           PT1H|  744|
|     2.0000|           PT1M|44640|
|    10.0000|          PT10M|44640|
|    10.0000|           PT1H|  744|
+-----------+---------------+-----+



## 6. Inspect coverage by series

Series validity periods restrict the nominal schedule. We do not expect measurements before a series opened, and a six-hourly series has a different schedule from a ten-minute series.

These counts are diagnostics. Frost's frequency can represent the shortest frequency when a schedule varies during the day. Validity windows here use `[validFrom, validTo)`. A nominal gap is not automatically evidence of failed ingestion. Unknown or failed series metadata remains visible through the metadata-failure report and null nominal expectations.


In [20]:
intervals, series_inputs, series_failures = load_series(
    bronze_root, [preview_station], preview_start, preview_end
)
print("Series metadata failures:", series_failures)
coverage = coverage_report(spark, accepted, intervals)
coverage.orderBy("source_id", "level_value", "time_resolution").show(30, truncate=False)


Series metadata failures: []


+---------+---------------+---------------+-----------+--------------+-------------------+----------+-----------+----------------+---------------------+---------------+------------------------+
|source_id|element_id     |time_resolution|time_offset|time_series_id|level_type         |level_unit|level_value|nominal_expected|accepted_observations|nominal_missing|outside_nominal_schedule|
+---------+---------------+---------------+-----------+--------------+-------------------+----------+-----------+----------------+---------------------+---------------+------------------------+
|SN18700:0|air_temperature|PT10M          |PT0H       |0             |height_above_ground|m         |2.0000     |4464            |4464                 |0              |0                       |
|SN18700:0|air_temperature|PT1H           |PT0H       |0             |height_above_ground|m         |2.0000     |744             |744                  |0              |0                       |
|SN18700:0|air_temperature|PT1

## 7. Clean up the example and run the full dataset

In [21]:
for frame in (normalized, accepted, quarantine, duplicates):
    frame.unpersist(blocking=True)
preview_directory.cleanup()

output = run(bronze_root, output_root, start_date, end_date, stations, spark)
print(output)


Silver temperature: processing 2024-01-01 to 2024-02-01


Silver temperature: processing 2024-02-01 to 2024-03-01


Silver temperature: processing 2024-03-01 to 2024-04-01


Silver temperature: processing 2024-04-01 to 2024-05-01


Silver temperature: processing 2024-05-01 to 2024-06-01


Silver temperature: processing 2024-06-01 to 2024-07-01


... (15 outputs left out)

Silver temperature: processing 2025-10-01 to 2025-11-01


Silver temperature: processing 2025-11-01 to 2025-12-01


Silver temperature: processing 2025-12-01 to 2026-01-01


Silver temperature output: /home/roland/UiS/H2026/group14/data/silver/temperature/7d02d24ce2f741c59ff156856fdd225f
/home/roland/UiS/H2026/group14/data/silver/temperature/7d02d24ce2f741c59ff156856fdd225f


## 8. Read the run report

`partial` means there were rejected observations or input/series-metadata failures. `coverage_requires_review` separately records empty station/months, nominal gaps, or observations outside nominal schedules. Neither `success` nor `partial` asserts that the stations represent all Norwegian price areas.

Each month's input rows must equal accepted + quarantined + duplicate rows. Empty responses contain zero observations, and failed inputs are recorded separately rather than counted as valid empty data.


In [22]:
run_info = json.loads((output / "run.json").read_text())
quality = json.loads((output / "quality.json").read_text())
print(json.dumps(run_info, indent=2))
print(json.dumps(quality["totals"], indent=2))
for month in quality["months"]:
    assert month["input_observations"] == (
        month["accepted_rows"] + month["quarantined_rows"] + month["duplicate_rows"]
    )


{
  "status": "partial",
  "rule_version": "temperature-v1",
  "started_at_utc": "2026-11-14T16:06:23.425976+00:00",
  "start_date": "2024-01-01",
  "end_date_exclusive": "2026-01-01",
  "date_timezone": "UTC",
  "stations": [
    "SN10055",
    "SN10070",
    "SN10380",
    "SN1070",
    "SN10800",
    "SN11000",
    "SN11060",
    "SN11125",
    "SN11340",
    "SN1135",
    "SN11440",
    "SN11463",
    "SN11500",
    "SN11690",
    "SN11790",
    "SN120",
    "SN12030",
    "SN12050",
    "SN12180",
    "SN12280",
    "SN12290",
    "SN12291",
    "SN12320",
    "SN12550",
    "SN12580",
    "SN12590",
    "SN12648",
    "SN12666",
    "SN12680",
    "SN12700",
    "SN12710",
    "SN12960",
    "SN12980",
    "SN1300",
    "SN13030",
    "SN13110",
    "SN13125",
    "SN13150",
    "SN13160",
    "SN13390",
    "SN13420",
    "SN13655",
    "SN13750",
    "SN13760",
    "SN1380",
    "SN14200",
    "SN14420",
    "SN15050",
    "SN15262",
    "SN15265",
    "SN15270",
    "SN15290",

## 9. Verify one saved month

Read back the first month and check saved disposition counts and key uniqueness.


In [23]:
first_month = quality["months"][0]
batch = output / ("month=" + first_month["month"])
saved = spark.read.parquet(str(batch / "accepted"))
saved_quarantine = spark.read.parquet(str(batch / "quarantine"))
saved_duplicates = spark.read.parquet(str(batch / "duplicates"))
assert saved.count() == first_month["accepted_rows"]
assert saved_quarantine.count() == first_month["quarantined_rows"]
assert saved_duplicates.count() == first_month["duplicate_rows"]
assert saved.groupBy(*BUSINESS_KEY).count().filter("count > 1").count() == 0
saved.select("station_id", "observed_utc", "temperature_c", "level_value", "time_resolution", "quality_uncertain").show(10, truncate=False)


+----------+-------------------+-------------+-----------+---------------+-----------------+
|station_id|observed_utc       |temperature_c|level_value|time_resolution|quality_uncertain|
+----------+-------------------+-------------+-----------+---------------+-----------------+
|SN10070   |2024-01-01 01:20:00|-9.400000    |2.0000     |PT10M          |false            |
|SN10070   |2024-01-01 02:30:00|-9.500000    |2.0000     |PT10M          |false            |
|SN10070   |2024-01-01 03:30:00|-9.400000    |2.0000     |PT10M          |false            |
|SN10070   |2024-01-01 05:30:00|-9.400000    |2.0000     |PT10M          |false            |
|SN10070   |2024-01-01 08:20:00|-9.500000    |2.0000     |PT10M          |false            |
|SN10070   |2024-01-01 09:00:00|-9.200000    |2.0000     |PT10M          |false            |
|SN10070   |2024-01-01 12:20:00|-8.700000    |2.0000     |PT10M          |false            |
|SN10070   |2024-01-01 13:20:00|-8.600000    |2.0000     |PT10M       

## 10. Stop the local Spark session


In [24]:
spark.stop()